In [1]:
import os
import requests
from datetime import datetime, timedelta
from pathlib import Path
import xarray as xr

# Model run date (YYYY, MM, DD)
RUN_DATE = datetime(2021, 1, 1)

# Model cycle hour (0, 6, 12, 18)
RUN_CYCLE = 0

# Forecast lead time in hours (0, 3, 6, ..., 384)
FORECAST_HOUR = 12

# Region bounding box (Germany)
LAT_MIN, LAT_MAX = 47, 55
LON_MIN, LON_MAX = 6, 15

valid_time = RUN_DATE + timedelta(hours=FORECAST_HOUR)

date_str = RUN_DATE.strftime("%Y%m%d")
cycle_str = f"{RUN_CYCLE:02d}"
forecast_str = f"{FORECAST_HOUR:03d}"

url = (
    f"https://noaa-gfs-bdp-pds.s3.amazonaws.com/"
    f"gfs.{date_str}/{cycle_str}/"
    f"gfs.t{cycle_str}z.sfluxgrbf{forecast_str}.grib2"
)

filename = f"gfs_{date_str}_{cycle_str}z_f{forecast_str}.grib2"

if not os.path.exists(filename):
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(filename, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

ds = xr.open_dataset(
    filename,
    engine="cfgrib",
    filter_by_keys={
        "typeOfLevel": "surface",
        "stepType": "avg"
    }
)

germany_ds = ds.sel(
    latitude=slice(LAT_MAX, LAT_MIN),
    longitude=slice(LON_MIN, LON_MAX)
)

zarr_path = f"germany_gfs_{date_str}_{cycle_str}z_f{forecast_str}.zarr"
germany_ds.to_zarr(zarr_path, mode="w", consolidated=True)

KeyboardInterrupt: 